# EDA — Customer Support on Twitter (selected brand)

Exploratory analysis only. All production logic lives in `src/`; this notebook inspects the data that `run_pipeline.py` also consumes, using the same `src.preprocessing` functions so the EDA never drifts out of sync with the pipeline.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing import load_raw_dataset, select_brand, build_conversation_pairs, clean_text

df = load_raw_dataset('../data/twcs.csv')
df.head()

In [ ]:
# Inbound vs outbound volume, and top candidate brand accounts
print(df['inbound'].value_counts())
outbound = df[df['inbound'].astype(str).str.lower() == 'false']
outbound['author_id'].value_counts().head(15)

In [ ]:
brand = select_brand(df, min_conversations=20)
pairs_df = build_conversation_pairs(df, brand)
print(brand, len(pairs_df), 'conversation pairs')
pairs_df.head()

In [ ]:
# Message length distribution (customer vs agent)
pairs_df['customer_len'] = pairs_df['customer_text'].str.split().apply(len)
pairs_df['agent_len'] = pairs_df['agent_text'].str.split().apply(len)
pairs_df[['customer_len', 'agent_len']].plot(kind='hist', alpha=0.6, bins=20, figsize=(8,4))
plt.title(f'Message length (words) - {brand}')
plt.xlabel('Words')
plt.show()

In [ ]:
# Most frequent words in customer messages, as a quick sanity check ahead of clustering (STEP 2)
from collections import Counter
tokens = ' '.join(pairs_df['customer_text']).lower().split()
Counter(tokens).most_common(25)

## Notes

- This notebook is exploratory only; intent discovery, classifier training, retrieval, generation, escalation, and evaluation are all implemented and executed via `run_pipeline.py` + `src/`, not here (per the assignment's "avoid notebooks except for EDA" instruction).
- If `data/twcs.csv` (the real Kaggle file) isn't present, `load_raw_dataset` automatically falls back to the synthetic sample at `data/sample_twcs.csv` -- see the warning it logs, and `README.md` for how to add the real file.